# exploracao_faturamento

Exploração da `bronze.tb_faturas` para levantar os problemas de qualidade e validar as regras do `silver_faturamento`.
**Não entra no Job:** só lê e exibe. Os resultados estão na última célula.

In [0]:
%run ./funcoes_comuns

In [0]:
CATALOGO = "workspace"
ft = ler_bronze(CATALOGO, "faturas")
print(f"{ft.count():,} linhas na Bronze")
ft = remover_cabecalho_repetido(ft)
print(f"{ft.count():,} linhas sem os cabeçalhos repetidos")

for c in [c for c in ft.columns if not c.startswith("_")]:
    perfilar_coluna(ft, c, n=5)

In [0]:
# Formatos de data e de valor (cada dígito vira 9)
for c in ["mes_referencia", "data_emissao", "data_vencimento", "data_pagamento",
          "valor_energia", "tributos", "valor_total"]:
    display(ft.groupBy(F.regexp_replace(c, r"\d", "9").alias(c)).count().orderBy(F.desc("count")))

In [0]:
# Data com hífen = mês-dia-ano: o 1º número nunca passa de 12 e, lendo assim,
# o vencimento (sempre yyyy-MM-dd) fica exatamente 12 dias depois da emissão em 100% das faturas
hifen = ft.filter(F.col("data_emissao").rlike(r"^\d{2}-\d{2}-\d{4}$"))
print(hifen.agg(F.count("*").alias("datas_com_hifen"),
                F.max(F.substring("data_emissao", 1, 2).cast("int")).alias("maior_1o_numero"),
                F.sum((F.substring("data_emissao", 4, 2).cast("int") > 12).cast("int")).alias("2o_numero_acima_de_12"))
           .first())
display(ft.groupBy(F.datediff(parse_data("data_vencimento"), parse_data("data_emissao")).alias("dias_ate_vencimento"))
          .count())

In [0]:
# "EM ABERTO com data de pagamento": a data é a sentinela 9999-12-31 (= sem pagamento), não um pagamento em atraso
pagamento = (F.when(F.col("data_pagamento").isNull(), "nula")
              .when(F.col("data_pagamento") == "9999-12-31", "9999-12-31").otherwise("data real"))
display(ft.groupBy("status", pagamento.alias("data_pagamento")).count().orderBy("status"))

In [0]:
# Depois do parse_decimal: nenhum valor perdido e total = energia + tributos em 100%
v = ft.select(*[parse_decimal(c).alias(c) for c in ["valor_energia", "tributos", "valor_total"]])
print(v.agg(F.count("*").alias("faturas"), F.count("valor_total").alias("convertidas"),
            F.sum((F.col("valor_total") == F.col("valor_energia") + F.col("tributos")).cast("int")).alias("total_confere"),
            F.min("valor_total").alias("min"), F.max("valor_total").alias("max")).first())

In [0]:
# Órfãs = ids FTX…, todas cópias exatas de uma fatura válida (só mudam id_fatura e id_uc).
# São elas que repetem UC+mês: depois de separar, (id_uc, mes_referencia) fica única
validas, orfas = separar_orfaos(ft, spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras"), "id_uc")
display(orfas.groupBy(F.col("id_fatura").startswith("FTX").alias("id_FTX")).count())

copia = [c for c in ft.columns if not c.startswith("_") and c not in ("id_fatura", "id_uc")]
iguais = [F.col(f"o.{c}").eqNullSafe(F.col(f"v.{c}")) for c in copia]
print("órfãs que copiam uma fatura válida:", orfas.alias("o").join(validas.alias("v"), iguais, "left_semi").count())

repetidos = lambda df: df.groupBy("id_uc", "mes_referencia").count().filter("count > 1").count()
print(f"pares UC+mês repetidos: {repetidos(ft)} | depois de separar as órfãs: {repetidos(validas)}")

In [0]:
# Bandeira: 4 valores em caixa mista e sempre uma única bandeira por mês (definida pela ANEEL)
display(ft.groupBy("bandeira_tarifaria").count())
print("meses com mais de uma bandeira:",
      ft.groupBy("mes_referencia").agg(F.countDistinct("bandeira_tarifaria").alias("n")).filter("n > 1").count())

## Resultados da exploração (faturas)

### Faturas (Bronze: 2.184.623 linhas)

| Achado | Quantidade | Regra na Silver |
|---|---|---|
| Cabeçalho do CSV repetido no meio do arquivo | 2 linhas | `remover_cabecalho_repetido` (REMOVIDO) |
| `data_emissao` em 4 formatos: `yyyy-MM-dd` (2.001.260), `dd/MM/yyyy` (61.291), `dd.MM.yyyy` (61.252) e `MM-dd-yyyy` (60.818). Hífen = mês-dia-ano: o 1º número nunca passa de 12, 38.615 têm o 2º acima de 12 e, lendo assim, o vencimento é emissão + 12 dias em 100% das faturas | 183.361 linhas | `parse_data` (CORRIGIDO) |
| `valor_total` no formato `R$ 1.234,56` (9%). `valor_energia` e `tributos` vêm sempre como `76.93` | 196.622 linhas | `parse_decimal` (CORRIGIDO). Depois: 0 nulos e total = energia + tributos em 100% |
| `data_pagamento = 9999-12-31` em faturas EM ABERTO. É a mesma sentinela de "sem data" da `data_retirada` dos medidores, **não** um pagamento em atraso. EM ABERTO = 113.835 com data nula + 61.333 com a sentinela; PAGA tem data real em 100% | 61.333 linhas | `parse_data` (sentinela → nulo); status mantido (CORRIGIDO). **A proposta do Padrão (status = PAGA + `flag_status_corrigido`) foi descartada:** marcaria como pagas 61 mil faturas que nunca foram pagas |
| Fatura de UC inexistente: todas com id `FTX…` e `id_uc` de 7 dígitos; todas são cópias exatas de faturas válidas (só mudam `id_fatura` e `id_uc`) | 32.285 linhas | `separar_orfaos` (QUARENTENA). Nenhuma receita real se perde |
| Pares UC+mês repetidos (todos entre as FTX) | 21 pares | resolvidos pela quarentena das órfãs |
| `bandeira_tarifaria` em caixa mista (`Verde`, `Amarela`, `Vermelha P1`, `Vermelha P2`) | 2.184.621 linhas | `padronizar_texto` (CORRIGIDO) |

**Limpos:** `id_fatura` único; `mes_referencia` sempre `yyyy-MM` (2022-07 a 2026-06); `data_vencimento` sempre `yyyy-MM-dd`; `status` só PAGA/EM ABERTO; nenhum espaço nas pontas; nenhuma duplicata idêntica; só `data_pagamento` tem nulo; sempre 1 bandeira por mês.

**Resultado esperado:** 2.152.336 faturas na Silver (45.000 UCs), 32.285 na quarentena e 2 removidas (2.184.623 = 2.152.336 + 32.285 + 2).

**Para a Gold:** toda fatura válida tem uma leitura com o mesmo `id_uc` + `mes_referencia` na Bronze de leituras (join direto); cerca de 8% das faturas de cada mês ficam EM ABERTO, de forma estável desde 2022.